# 답변 품질 검토

In [1]:
from pathlib import Path
import json, sys
import pandas as pd
try:
    from IPython.display import display, HTML
except ImportError:
    class HTML:
        def __init__(self, data): self.data = data
        def _repr_html_(self): return self.data
    def display(value): print(value)

working_dir = Path.cwd().resolve()
candidates = [working_dir, working_dir / 'code', working_dir.parent, working_dir.parent / 'code']
CODE_ROOT = next((candidate for candidate in candidates if (candidate / 'rag').is_dir() and (candidate / 'notebooks').is_dir()), None)
if CODE_ROOT is None:
    raise FileNotFoundError('프로젝트 폴더, code/ 또는 code/notebooks/에서 실행하세요.')
sys.path.insert(0, str(CODE_ROOT))

RUN_LOCAL = False  # True로 바꾸기 전 입력 자료·모델·장치·저장 위치를 확인
print('기본 모드: 셀에 명시한 저장된 평가 집계 확인. API 호출·모델 다운로드·학습 없음.')

기본 모드: 셀에 명시한 저장된 평가 집계 확인. API 호출·모델 다운로드·학습 없음.


In [2]:
# 저장된 평가 집계와 실행 설정. 원본 값을 그대로 옮겼으며 새로 평가하지 않음.
answers = {'fact_recall': {'n': 154,
                 'mean': 0.8614718614718614,
                 'ci95': [0.816017316017316, 0.9037067099567097]},
 'false_web_switch': {'n': 154,
                      'mean': 0.09090909090909091,
                      'ci95': [0.045454545454545456, 0.13636363636363635]},
 'missed_web_switch': {'n': 97,
                       'mean': 0.05154639175257732,
                       'ci95': [0.010309278350515464, 0.10309278350515463]},
 'normal_as_out_of_scope': {'n': 260, 'count': 0},
 'out_of_scope': {'k': 17, 'n': 20, 'rate': 0.85, 'cp95': [0.621073173454686, 0.9679290628145363]},
 'extra_information': {'k': 0,
                       'n': 20,
                       'rule': '0/20',
                       'pass_': True,
                       'cp95': [0.0, 0.16843347098308528],
                       'n_bullets': 29,
                       'n_answers_with_ambiguous': 0},
 'body_length': {'n': 154, 'median_chars': 272.0},
 'human_support': {'n_answers': 40,
                   'n_sentences': 240,
                   'answers_with_outside': 0,
                   'rate': 0.0,
                   'cp95': [0.0, 0.08809730287880237],
                   'n_outside_sentences': 0,
                   'answers_with_ambiguous': 0,
                   'n_ambiguous_sentences': 0},
 'human_original_criteria': {'n_answers': 40,
                             'n_sentences': 240,
                             'answers_with_outside': 1,
                             'rate': 0.025,
                             'cp95': [0.0006327449320494191, 0.13158585848276572],
                             'n_outside_sentences': 1,
                             'answers_with_ambiguous': 8,
                             'n_ambiguous_sentences': 8,
                             'incl_ambiguous_as_outside': {'k': 9,
                                                           'cp95': [0.10839663898407201,
                                                                    0.3845116773034227]}},
 'judge_agreement': {'n_answers': 30,
                     'n_facts': 65,
                     'exact_agreement': [0.9230769230769231,
                                         0.8591046277665996,
                                         0.9836065573770492],
                     'kappa': [0.6364653243847875, 0.17360967630854, 0.8994972159893622],
                     'recall_judge_minus_human': [-0.061111111111111116,
                                                  -0.15,
                                                  0.011111111111111112]}}
display(pd.DataFrame([{
    'metric':name, 'questions':answers[name]['n'], 'mean':answers[name]['mean'],
    'ci_low':answers[name]['ci95'][0], 'ci_high':answers[name]['ci95'][1]
} for name in ['fact_recall','false_web_switch','missed_web_switch']]).set_index('metric').round(3))

,questions,mean,ci_low,ci_high
metric,,,,
fact_recall,154,0.861,0.816,0.904
false_web_switch,154,0.091,0.045,0.136
missed_web_switch,97,0.052,0.010,0.103


## 사람 검토와 기준 차이

In [3]:
display(pd.DataFrame([answers['human_support'], answers['human_original_criteria']], index=['user_confirmed_exceptions','original_criteria']).drop(columns=['incl_ambiguous_as_outside'], errors='ignore'))
display(pd.Series(answers['human_original_criteria']['incl_ambiguous_as_outside'], name='ambiguous_included').to_frame())

,n_answers,n_sentences,answers_with_outside,rate,cp95,n_outside_sentences,answers_with_ambiguous,n_ambiguous_sentences
user_confirmed_exceptions,40,240,0,0.000,"[0.0, 0.08809730287880237]",0,0,0
original_criteria,40,240,1,0.025,"[0.0006327449320494191, 0.13158585848276572]",1,8,8


,ambiguous_included
k,9
cp95,"[0.10839663898407201, 0.3845116773034227]"


## 범위 밖 질문과 판정 신뢰도

In [4]:
display(pd.Series(answers['out_of_scope'], name='out_of_scope').to_frame())
display(pd.Series(answers['judge_agreement'], name='judge_agreement').to_frame())

,out_of_scope
k,17
n,20
rate,0.85
cp95,"[0.621073173454686, 0.9679290628145363]"


,judge_agreement
n_answers,30
n_facts,65
exact_agreement,"[0.9230769230769231, 0.8591046277665996, 0.983..."
kappa,"[0.6364653243847875, 0.17360967630854, 0.89949..."
recall_judge_minus_human,"[-0.061111111111111116, -0.15, 0.0111111111111..."
